# Basic

Greedy token selection with the **status-dependent decay**: a hop into a primitive concept costs $\lambda$,
a hop into a fully defined concept costs nothing (`prim_fd_tokenizer.StatusLazyGreedyTokenSelector`).
Results go to `config.CandidateLists.path_greedy_tree` (`greedy_tree_candidates_prim_fd/`), one file per $\lambda$.

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, where src/ lives

import config_prim_fd as config             # found in decay_with_prim_fd/
import prim_fd_tokenizer as pf              # status-dependent coverage, selection and distance score

import os
import time

import polars as pl
import numpy as np
import tqdm

import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct

# graph is unchanged: notebook 1 writes it to processed_graph/, graph_fct reads it from there
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()

# concept status (primitive / fully defined) for every node of the graph
is_primitive = pf.load_is_primitive(config.ProcessedGraph.concept_and_status, nodes=combined_subgraphs.nodes())

D = config.TokenizerParam().max_dist_candidate
out_path = config.CandidateLists().path_greedy_tree
assert "prim_fd" in out_path, "would overwrite the hop-based greedy candidates"
os.makedirs(out_path, exist_ok=True)

status: 24,515 primitive / 51,449 fully defined (0 without status, treated as primitive)


# greedy selection, decay on primitive hop targets

In [3]:
lambdas_list = config.TokenizerParam().lambdas   # 0.0-1.0 step 0.1, plus 0.91-0.99

for lam in tqdm.tqdm(lambdas_list):
    t0 = time.time()
    selector = pf.StatusLazyGreedyTokenSelector(combined_subgraphs, mapped_ids, D=D,
                                                is_primitive=is_primitive, lam=lam)
    # verbose=False: one printed line per selected token made the original notebook ~40 MB
    history = selector.select(k=int(len(mapped_ids) / 2), n_jobs=-1, verbose=False)
    df_history = (
        pl.DataFrame(history, schema=["token", "gain", "cumulative_score"], orient="row")
        .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .with_row_index())
    df_history.write_parquet(f"{out_path}{lam}.parquet")
    print(f"lambda={lam}: {len(history):,} tokens, F_D={selector.current_score:.4f}, {(time.time() - t0) / 60:.1f} min")

  5%|▌         | 1/20 [05:58<1:53:32, 358.53s/it]

lambda=0.0: 23,075 tokens, F_D=0.6736, 6.0 min


 10%|█         | 2/20 [13:49<2:07:23, 424.63s/it]

lambda=0.1: 23,075 tokens, F_D=0.6920, 7.8 min


 15%|█▌        | 3/20 [22:01<2:09:02, 455.46s/it]

...

lambda=0.98: 23,075 tokens, F_D=0.9920, 16.3 min


 95%|█████████▌| 19/20 [3:53:48<15:41, 941.33s/it]

lambda=0.99: 23,075 tokens, F_D=0.9960, 16.7 min


100%|██████████| 20/20 [4:06:28<00:00, 739.43s/it]

lambda=1.0: 23,075 tokens, F_D=1.0000, 12.7 min


# sanity check

With $\lambda = 1$ every weight is 1, so this selection should match the hop-based $\lambda = 1$ run,
as long as that file was made with the current code and the same `mapped_concepts.parquet`.

In [4]:
import src.graph_tokenizer_gd_tree_dev.config as config_hop

new_1 = pl.read_parquet(f"{out_path}1.0.parquet")["token"].to_list()
old_1 = pl.read_parquet(f"{config_hop.CandidateLists().path_greedy_tree}1.0.parquet")["token"].to_list()
n = min(len(new_1), len(old_1))
first_diff = next((i for i in range(n) if new_1[i] != old_1[i]), None)
print(f"lambda = 1: {len(new_1):,} vs {len(old_1):,} tokens; same order up to rank {first_diff if first_diff is not None else n:,}")

lambda = 1: 23,075 vs 23,075 tokens; same order up to rank 544
